In [ ]:
import pandas as pd
import numpy as np
import nfl_data_py as nfl
from pygam import LinearGAM, s, f, te
import matplotlib.pyplot as plt
import data_processing_functions as helpy

In [2]:
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
recieving_2024 = helpy.get_year_table(2024)
recieving_2023 = helpy.get_year_table(2023)
recieving_2022 = helpy.get_year_table(2022)
recieving_2021 = helpy.get_year_table(2021)
recieving_2020 = helpy.get_year_table(2020)
recieving_2019 = helpy.get_year_table(2019)
recieving_2018 = helpy.get_year_table(2018)

In [5]:
recieving = helpy.merge_datasets([recieving_2018, recieving_2019, recieving_2020, 
                                  recieving_2021, recieving_2022, recieving_2023, recieving_2024])

In [6]:
X = recieving[['man_yprr', 'man_yards', 'zone_yprr', 'zone_yards', 'deep_yards', 'medium_yards', 'short_yards', 
                         'behind_los_yards', 'var_depth', 'drop_rate', 'wide_rate', 'man_avg_depth_of_target', 
                         'zone_avg_depth_of_target', 'contested_receptions', 'contested_catch_rate']]
y = recieving['pick']

In [8]:
recieving

,player,player_id,position,team_name,contested_receptions,contested_catch_rate,targets,yards,touchdowns,avg_depth_of_target,drop_rate,wide_rate,behind_los_yards,short_yards,medium_yards,deep_yards,man_targets,man_yprr,man_avg_depth_of_target,man_yards,zone_targets,zone_yprr,zone_avg_depth_of_target,zone_yards,pick,pfr_player_name,var_depth,is_power_four
0,Andy Isabella,47448,WR,UMASS,9,36.0,146,1696,13,12.1,5.6,55.8,170,351,470,705,39,3.89,14.2,428,100,4.47,11.5,1216,62,Andy Isabella,50307.333333,False
1,John Ursua,26586,WR,HAWAII,10,47.6,145,1343,16,12.1,12.7,3.0,21,513,386,423,38,2.81,11.4,427,94,2.27,12.9,886,236,John Ursua,46874.250000,False
2,Dillon Mitchell,42338,WR,OREGON,10,27.8,130,1184,10,14.6,9.6,76.5,90,321,264,509,59,3.65,14.6,533,66,2.58,14.0,618,239,Dillon Mitchell,29818.000000,False
3,KeeSean Johnson,39587,WR,FRESNO ST,11,47.8,129,1345,8,11.1,6.8,69.6,101,389,314,541,29,2.82,8.4,282,98,3.28,11.9,1052,174,KeeSean Johnson,33514.250000,False
4,Kelvin Harmon,47931,WR,NC STATE,17,56.7,117,1186,7,15.1,4.7,96.4,31,302,387,466,30,3.27,12.4,363,81,2.85,15.9,804,206,Kelvin Harmon,35813.666667,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,Jaylin Lane,121811,WR,VA TECH,1,16.7,58,466,2,7.5,7.3,13.7,96,164,115,91,20,1.27,11.9,99,29,2.16,4.2,326,128,Jaylin Lane,1109.666667,False
26,Jimmy Horn,144483,WR,COLORADO,2,33.3,53,441,1,7.8,5.1,5.6,74,119,107,141,8,1.79,3.9,77,37,1.72,8.1,307,208,Jimmy Horn,782.250000,False
27,Tory Horton,128232,WR,COLO STATE,3,50.0,38,331,1,13.5,3.8,64.7,8,98,115,110,10,3.50,14.2,91,23,3.62,14.4,192,166,Tory Horton,2534.250000,False
28,Isaac TeSlaa,172430,WR,ARKANSAS,5,71.4,36,532,3,14.3,0.0,20.6,17,76,162,277,8,1.01,17.1,68,18,1.81,11.9,289,70,Isaac TeSlaa,12760.666667,False


This below should be mostly monotonic eventually but without more data it just won't work :0

In [38]:
gam = LinearGAM(te(0, 1, 11, n_splines=5) + te(2, 3, 12, n_splines=5) + s(4, n_splines=5) + s(5, n_splines=5) + s(6, n_splines=5) + 
                s(7, n_splines=5) + s(8, n_splines=5) + s(9, n_splines=5) + s(10, n_splines=5) + te(13, 14, n_splines=5))
gam.fit(X, y)

LinearGAM(callbacks=[Deviance(), Diffs()], fit_intercept=True, 
   max_iter=100, scale=None, 
   terms=te(0, 1, 11) + te(2, 3, 12) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9) + s(10) + te(13, 14) + intercept,
   tol=0.0001, verbose=False)

In [39]:
gam.summary()

LinearGAM                                                                                                 
=============================================== ==========================================================
Distribution:                        NormalDist Effective DoF:                                     10.1349
Link Function:                     IdentityLink Log Likelihood:                                 -1095.6204
Number of Samples:                          195 AIC:                                             2213.5106
                                                AICc:                                            2214.9884
                                                GCV:                                             5161.3268
                                                Scale:                                             68.4165
                                                Pseudo R-Squared:                                   0.2314
Feature Function                  Lam

/tmp/ipykernel_16419/3358381670.py:1: UserWarning: KNOWN BUG: p-values computed in this summary are likely much smaller than they should be. 
 
Please do not make inferences based on these values! 

Collaborate on a solution, and stay up to date at: 
github.com/dswah/pyGAM/issues/163 

  gam.summary()


In [40]:
predicted_slot = gam.predict(X)

In [41]:
recieving['predicted_slot'] = predicted_slot

In [42]:
recieving

,player,player_id,position,team_name,contested_receptions,contested_catch_rate,targets,yards,touchdowns,avg_depth_of_target,drop_rate,wide_rate,behind_los_yards,short_yards,medium_yards,deep_yards,man_targets,man_yprr,man_avg_depth_of_target,man_yards,zone_targets,zone_yprr,zone_avg_depth_of_target,zone_yards,pick,pfr_player_name,var_depth,predicted_slot
0,Andy Isabella,47448,WR,UMASS,9,36.0,146,1696,13,12.1,5.6,55.8,170,351,470,705,39,3.89,14.2,428,100,4.47,11.5,1216,62,Andy Isabella,50307.333333,45.364313
1,John Ursua,26586,WR,HAWAII,10,47.6,145,1343,16,12.1,12.7,3.0,21,513,386,423,38,2.81,11.4,427,94,2.27,12.9,886,236,John Ursua,46874.250000,129.034807
2,Dillon Mitchell,42338,WR,OREGON,10,27.8,130,1184,10,14.6,9.6,76.5,90,321,264,509,59,3.65,14.6,533,66,2.58,14.0,618,239,Dillon Mitchell,29818.000000,99.758314
3,KeeSean Johnson,39587,WR,FRESNO ST,11,47.8,129,1345,8,11.1,6.8,69.6,101,389,314,541,29,2.82,8.4,282,98,3.28,11.9,1052,174,KeeSean Johnson,33514.250000,79.150324
4,Kelvin Harmon,47931,WR,NC STATE,17,56.7,117,1186,7,15.1,4.7,96.4,31,302,387,466,30,3.27,12.4,363,81,2.85,15.9,804,206,Kelvin Harmon,35813.666667,99.641634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,Jaylin Lane,121811,WR,VA TECH,1,16.7,58,466,2,7.5,7.3,13.7,96,164,115,91,20,1.27,11.9,99,29,2.16,4.2,326,128,Jaylin Lane,1109.666667,158.830752
26,Jimmy Horn,144483,WR,COLORADO,2,33.3,53,441,1,7.8,5.1,5.6,74,119,107,141,8,1.79,3.9,77,37,1.72,8.1,307,208,Jimmy Horn,782.250000,157.305915
27,Tory Horton,128232,WR,COLO STATE,3,50.0,38,331,1,13.5,3.8,64.7,8,98,115,110,10,3.50,14.2,91,23,3.62,14.4,192,166,Tory Horton,2534.250000,138.857492
28,Isaac TeSlaa,172430,WR,ARKANSAS,5,71.4,36,532,3,14.3,0.0,20.6,17,76,162,277,8,1.01,17.1,68,18,1.81,11.9,289,70,Isaac TeSlaa,12760.666667,135.250215


In [16]:
recieving[recieving['pfr_player_name'] == 'Tetairoa McMillan']

,player,player_id,position,team_name,contested_receptions,contested_catch_rate,targets,yards,touchdowns,avg_depth_of_target,...,man_avg_depth_of_target,man_yards,zone_targets,zone_yprr,zone_avg_depth_of_target,zone_yards,pick,pfr_player_name,var_depth,predicted_slot_yprr
1,Tetairoa McMillan,158735,WR,ARIZONA,18,60.0,130,1316,8,13.7,...,13.2,481,69,2.8,14.2,755,8,Tetairoa McMillan,49904.666667,62.617408
